# Impute some field
**Warning**: It required to **run this file and wait for about 6 hours** in order to complete it. It's an advice to use already cleaned version provided by our team.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))
import time
import requests
import json
import re
import os
import pandas as pd
from llama_index.llms.ollama import Ollama
from llama_index.core.llms import ChatMessage
from utils.preprocess.imputation.crawler import fetch_steam_app_data

os.makedirs('data/checkpoint/', exist_ok=True)
os.makedirs('data/output/', exist_ok=True)

In [ ]:
# if you run this first time, use `data/origin/SteamGames.csv`
# if you have check point and need to rerun, use `data/checkpoint/fallback_check_point.csv`
DATA_PATH = 'data/checkpoint/fallback_check_point.csv'
df = pd.read_csv(DATA_PATH)

### Impuate Description

In [10]:
target_cols = ["Description", "Genres", "OsRequirement", "MemoryRequirement", "CpuRequirement"]
missing_mask = df[target_cols].isna().any(axis=1)
missing_indices = df[missing_mask].index


print(f"Found {len(missing_indices)} rows needing API enrichment.")

session = requests.Session()
print(f'Request session complete')
for count, idx in enumerate(missing_indices, start=1):
    appId = int(df.loc[idx, 'Appid'])
    crawled = fetch_steam_app_data(appId, session)
    print(f'Fetch {idx} complete')

    if crawled:
        for col in target_cols:
            if pd.isna(df.loc[idx, col]) and crawled.get(col) is not None:
                df.loc[idx, col] = crawled[col]

    time.sleep(5.0)

    if count % 100 == 0:
        # checkpoint
        df.to_csv('data/checkpoint/fallback_check_point.csv', index=False)
        print(f"Checkpointed {count}/{len(missing_indices)} rows.")
print(f"Complete imputate!")

/home/leezy/.conda/envs/ds-steam-game/lib/python3.12/site-packages/certifi/cacert.pem
Found 3747 rows needing API enrichment.
Request session complete
Fetch 34 complete
Fetch 37 complete
Fetch 40 complete
Fetch 90 complete
Fetch 145 complete
Fetch 182 complete
Fetch 185 complete
Fetch 190 complete
Fetch 209 complete
Fetch 211 complete
Fetch 213 complete
Fetch 220 complete
Fetch 229 complete
Fetch 230 complete
Fetch 284 complete
Fetch 289 complete
Fetch 311 complete
Fetch 319 complete
Fetch 340 complete
Fetch 354 complete
Fetch 374 complete
Fetch 391 complete
Fetch 442 complete
Fetch 445 complete
Fetch 456 complete
Fetch 458 complete
Fetch 482 complete
Fetch 508 complete
Fetch 517 complete
Fetch 520 complete
Fetch 654 complete
Fetch 684 complete
Fetch 710 complete
Fetch 739 complete
Fetch 759 complete
Fetch 768 complete
Fetch 778 complete
Fetch 794 complete
Fetch 796 complete
Fetch 800 complete
Fetch 813 complete
Fetch 824 complete
Fetch 826 complete
Fetch 840 complete
Fetch 841 complet

### Fallback: fill out any missing remains using LLM

In [11]:
# 1. Initialize Ollama LLM
llm = Ollama(
    model="qwen3.5:2b",
    base_url="http://localhost:11434",
    request_timeout=600.0,
    json_mode=True,
    temperature=0.2,  # Low temperature keeps descriptions factual and grounded
)

# 2. Complete System & User Prompts (Notice doubled {{ }} for literal JSON braces)
SYSTEM_PROMPT = """
You are a video game catalog archivist. Your task is to write a concise, factual, 2-to-4 sentence Steam store description based strictly on the provided game metadata.
Rules:
- Focus on the game's core genre, themes, and gameplay style implied by the metadata.
- Do NOT invent specific story character names, fake release dates, or unverified features.
- Output ONLY a valid JSON object with a single key "description".
""".strip()

USER_PROMPT = """
Write a short and sweet Steam store description for "{game_name}" (maximum 150 words).
Name: {game_name}
Type: {game_type}
Genres: {game_genres}
Tags: {game_tags}
Developer: {game_developer}
Publisher: {game_publisher}

Strictly follow the JSON output format below:
{{
    "description": "Your concise game description here."
}}
""".strip()

In [12]:
keep_features = ["game", "dlc"]
df = df[df['Type'].isin(keep_features)].reset_index(drop=True)

# Check for any rows in Description feature missing
missing_mask = df['Description'].isna()
missing_indices = df[missing_mask].index

In [13]:
def safe_val(val, default="Unknown") -> str:
    """Helper to convert NaN/None cells to a clean string before formatting."""
    return str(val).strip() if pd.notna(val) and str(val).strip() else default

# 3. Enrich missing rows
missing_indices = df[df["Description"].isna()].index
print(f"Found {len(missing_indices)} rows needing LLM generation enrichment.")

for count, index in enumerate(missing_indices, start=1):
    row = df.loc[index]
    
    # Fill out the user prompt using safe_val to handle any NaN columns
    formatted_user_prompt = USER_PROMPT.format(
        game_name=safe_val(row.get("Name")),
        game_type=safe_val(row.get("Type"), default="game"),
        game_genres=safe_val(row.get("Genres")),
        game_tags=safe_val(row.get("Tags")),
        game_developer=safe_val(row.get("Developers")),
        game_publisher=safe_val(row.get("Publishers")),
    )

    messages = [
        ChatMessage(role="system", content=SYSTEM_PROMPT),
        ChatMessage(role="user", content=formatted_user_prompt),
    ]

    try:
        # Use .chat() for ChatMessage lists
        response = llm.chat(messages)
        raw_content = response.message.content.strip()
        
        # Strip <think>...</think> blocks just in case the model outputs reasoning tags
        clean_json_str = re.sub(r"<think>.*?</think>", "", raw_content, flags=re.DOTALL).strip()
        
        # Parse JSON string into a Python dict
        data = json.loads(clean_json_str)
        description_text = data.get("description", "").strip()
        
        if description_text:
            df.loc[index, "Description"] = description_text
            print(f"[{count}/{len(missing_indices)}] Generated description for: {row.get('Name')}")
        else:
            print(f"[{count}/{len(missing_indices)}] Warning: Empty description returned for index {index}")

    except Exception as e:
        print(f"[{count}/{len(missing_indices)}] Error on index {index} ({row.get('Name')}): {e}")

    # Save checkpoint every 10 rows (and on the final row)
    if count % 10 == 0 or count == len(missing_indices):
        df.to_csv("data/checkpoint/fallback_check_point.csv", index=False)
        print(f"Checkpointed {count}/{len(missing_indices)} rows.")

df.to_csv("data/output/SteamGames_Clean_1.csv", index=False)
print("Generation impute complete.")

Found 2 rows needing LLM generation enrichment.
[1/2] Generated description for: The Evil Within: The Executioner
[2/2] Error on index 24726 (TOXIKK - [UPGRADE] - Free Edition to FULL GAME): timed out
Checkpointed 2/2 rows.
Generation impute complete.
